In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer 
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

Import the datas.

In [21]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
df_orig = (
    pd.read_csv('data/personality_datasert.csv')
    .rename(columns={'Personality': 'match_p'})
    .drop_duplicates(['Time_spent_Alone', 'Stage_fear', 'Social_event_attendance',
                      'Going_outside', 'Drained_after_socializing', 'Friends_circle_size',
                      'Post_frequency'])
)
print(train_df.dtypes, df_orig.dtypes)

id                             int64
Time_spent_Alone             float64
Stage_fear                    object
Social_event_attendance      float64
Going_outside                float64
Drained_after_socializing     object
Friends_circle_size          float64
Post_frequency               float64
Personality                   object
dtype: object Time_spent_Alone             float64
Stage_fear                    object
Social_event_attendance      float64
Going_outside                float64
Drained_after_socializing     object
Friends_circle_size          float64
Post_frequency               float64
match_p                       object
dtype: object


In [40]:
train_df = train_df.merge(df_orig, how='left')
test_df = test_df.merge(df_orig, how='left')
train_df

,id,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality,match_p
0,0,0.0,No,6.0,4.0,No,15.0,5.0,Extrovert,NaN
1,1,1.0,No,7.0,3.0,No,10.0,8.0,Extrovert,NaN
2,2,6.0,Yes,1.0,0.0,NaN,3.0,0.0,Introvert,NaN
3,3,3.0,No,7.0,3.0,No,11.0,5.0,Extrovert,NaN
4,4,1.0,No,4.0,4.0,No,13.0,NaN,Extrovert,NaN
...,...,...,...,...,...,...,...,...,...,...
18519,18519,3.0,No,7.0,3.0,No,9.0,7.0,Extrovert,NaN
18520,18520,1.0,NaN,6.0,7.0,No,6.0,5.0,Extrovert,NaN
18521,18521,7.0,Yes,1.0,1.0,Yes,1.0,NaN,Introvert,NaN
18522,18522,NaN,Yes,1.0,0.0,Yes,5.0,2.0,Introvert,NaN


In [16]:
TARGET_COLUMN = 'Personality'
mapping = {'Introvert': 0, 'Extrovert': 1}
y = train_df[TARGET_COLUMN].map(mapping)

X = train_df.drop(["id", TARGET_COLUMN], axis=1)
X_test = test_df.drop("id", axis=1)

X['match_p_is_null'] = X['match_p'].isna().astype(int)
X_test['match_p_is_null'] = X_test['match_p'].isna().astype(int)

X_test = X_test[X.columns]

categorical_features = X.select_dtypes(include=['object', 'category']).columns
numerical_features = X.select_dtypes(include=np.number).columns 

X[categorical_features] = X[categorical_features].fillna('Unknown')
X_test[categorical_features] = X_test[categorical_features].fillna('Unknown')

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')) # Điền giá trị thiếu bằng trung bình
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [29]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, device='cuda')
lgbm_clf = LGBMClassifier(random_state=42)
catboost_clf = CatBoostClassifier(random_state=42)

param_grid_xgb = {
    'classifier__n_estimators': [200, 300, 400],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [5, 7, 10],
    'classifier__subsample': [0.7, 0.8],
    'classifier__colsample_bytree': [0.7, 0.8],
}
param_grid_lgbm = {
    'classifier__n_estimators': [200, 300, 400],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__num_leaves': [20, 31, 40],
    'classifier__subsample': [0.7, 0.8],
    'classifier__colsample_bytree': [0.7, 0.8],
}

param_grid_catboost = {
    'classifier__iterations': [200, 300],
    'classifier__learning_rate': [0.01, 0.05],
    'classifier__depth': [4, 6, 8]
}

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_clf)
])
lgbm_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', lgbm_clf)
])

catboost_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', catboost_clf)
])

In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Optimizing XGBoost...")
grid_search_xgb = GridSearchCV(xgb_pipeline, param_grid_xgb, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_xgb.fit(X, y)

print("\nOptimizing LightGBM...")
grid_search_lgbm = GridSearchCV(lgbm_pipeline, param_grid_lgbm, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_lgbm.fit(X, y)

print("\nBest XGBoost params:", grid_search_xgb.best_params_)
print("Best XGBoost score:", grid_search_xgb.best_score_)
print("\nBest LightGBM params:", grid_search_lgbm.best_params_)
print("Best LightGBM score:", grid_search_lgbm.best_score_)

Optimizing XGBoost...
Fitting 5 folds for each of 108 candidates, totalling 540 fits


/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:01:03] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:01:03] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:01:03] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:01:03] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/o


Optimizing LightGBM...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10960, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002262 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 9
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003248 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number 

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003600 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494
[LightGBM] [Info] Number of positive: 10960, number of negative: 3860
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.070948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 14820, number of used features: 9
[LightGBM] [Info] [bin

...

[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.015061 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.043494
[LightGBM] [Info] Start training from score 1.043494


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 10960, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001409 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 14820, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739541 -> initscore=1.043585
[LightGBM] [Info] Start training from score 1.043585


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/pytho

[LightGBM] [Info] Number of positive: 13699, number of negative: 4825
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000463 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 18524, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739527 -> initscore=1.043512
[LightGBM] [Info] Start training from score 1.043512

Best XGBoost params: {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 7, 'classifier__n_estimators': 200, 'classifier__subsample': 0.7}
Best XGBoost score: 0.9702549091619229

Best LightGBM params: {'classifier__colsample_bytree': 0.7, 'classifier__learning_rate': 0.05, 'classifier__n_estimators': 300, 'classifier__num_leaves': 20, 'classifier__subsample': 0.7}
Best LightGBM score: 0.96998498905

In [30]:
print('Catboost optimizing')
grid_search_catboost = GridSearchCV(catboost_pipeline,param_grid=param_grid_catboost, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search_catboost.fit(X,y)

Catboost optimizing
Fitting 5 folds for each of 12 candidates, totalling 60 fits
0:	learn: 0.6763690	total: 61.1ms	remaining: 12.2s
0:	learn: 0.6767120	total: 61ms	remaining: 12.1s
0:	learn: 0.6146117	total: 60.9ms	remaining: 12.1s
0:	learn: 0.6122698	total: 59.6ms	remaining: 11.9s
0:	learn: 0.6766920	total: 59.4ms	remaining: 11.8s
1:	learn: 0.6604016	total: 65.3ms	remaining: 6.46s
1:	learn: 0.5457391	total: 65.2ms	remaining: 6.45s
1:	learn: 0.5443252	total: 64ms	remaining: 6.34s
1:	learn: 0.6606633	total: 64.8ms	remaining: 6.42s
1:	learn: 0.6601231	total: 67.7ms	remaining: 6.7s
2:	learn: 0.4852233	total: 68.6ms	remaining: 4.51s
2:	learn: 0.6448056	total: 69.5ms	remaining: 4.57s
2:	learn: 0.4861429	total: 68.8ms	remaining: 4.52s
2:	learn: 0.6453399	total: 68.7ms	remaining: 4.51s
2:	learn: 0.6443848	total: 72.5ms	remaining: 4.76s
3:	learn: 0.4354816	total: 72.4ms	remaining: 3.55s
3:	learn: 0.6305730	total: 71.4ms	remaining: 3.5s
3:	learn: 0.6294289	total: 74ms	remaining: 3.63s
3:	learn:

,estimator,Pipeline(step...x1450c5e90>)])
,param_grid,"{'classifier__depth': [4, 6, ...], 'classifier__iterations': [200, 300], 'classifier__learning_rate': [0.01, 0.05]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [32]:
best_xgb = grid_search_xgb.best_estimator_
best_lgbm = grid_search_lgbm.best_estimator_
best_catboost = grid_search_catboost.best_estimator_
xgb_weight = grid_search_xgb.best_score_
lgbm_weight = grid_search_lgbm.best_score_
catboost_weight = grid_search_catboost.best_score_

ensemble = VotingClassifier(
    estimators=[
        ('xgb', best_xgb),
        ('lgbm', best_lgbm),
        ('catboost', best_catboost)
    ],
    voting='soft', 
    weights=[xgb_weight, lgbm_weight, catboost_weight],  
    n_jobs=-1
)

print("\nTraining ensemble model...")
ensemble.fit(X, y)


Training ensemble model...
0:	learn: 0.6129472	total: 2.79ms	remaining: 835ms
1:	learn: 0.5456588	total: 5.55ms	remaining: 828ms
2:	learn: 0.4866700	total: 8.01ms	remaining: 793ms
3:	learn: 0.4376015	total: 10.1ms	remaining: 748ms
4:	learn: 0.3952016	total: 12.4ms	remaining: 734ms
5:	learn: 0.3603733	total: 14.3ms	remaining: 703ms
6:	learn: 0.3295251	total: 16.4ms	remaining: 687ms
7:	learn: 0.3023780	total: 18.3ms	remaining: 667ms
8:	learn: 0.2792319	total: 20.6ms	remaining: 667ms
9:	learn: 0.2593210	total: 22.8ms	remaining: 661ms
10:	learn: 0.2425994	total: 24.9ms	remaining: 654ms
11:	learn: 0.2278393	total: 27.1ms	remaining: 651ms
12:	learn: 0.2157283	total: 29.3ms	remaining: 647ms
13:	learn: 0.2047842	total: 31.3ms	remaining: 639ms
14:	learn: 0.1957355	total: 33.7ms	remaining: 641ms
15:	learn: 0.1875274	total: 35.9ms	remaining: 638ms
16:	learn: 0.1804807	total: 38.2ms	remaining: 636ms
17:	learn: 0.1745573	total: 40.5ms	remaining: 634ms
18:	learn: 0.1690005	total: 42.5ms	remaining: 

/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:29:02] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:29:02] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


36:	learn: 0.1318813	total: 84.7ms	remaining: 602ms
37:	learn: 0.1311955	total: 87.4ms	remaining: 603ms
38:	learn: 0.1306532	total: 90.2ms	remaining: 604ms
39:	learn: 0.1300934	total: 92.2ms	remaining: 600ms
40:	learn: 0.1296092	total: 95.6ms	remaining: 604ms
41:	learn: 0.1291060	total: 98.4ms	remaining: 604ms
42:	learn: 0.1287363	total: 101ms	remaining: 603ms
43:	learn: 0.1283366	total: 103ms	remaining: 601ms
44:	learn: 0.1279593	total: 106ms	remaining: 599ms
45:	learn: 0.1276546	total: 108ms	remaining: 598ms
46:	learn: 0.1273858	total: 111ms	remaining: 600ms
47:	learn: 0.1271681	total: 114ms	remaining: 599ms
48:	learn: 0.1269419	total: 117ms	remaining: 597ms
49:	learn: 0.1267283	total: 119ms	remaining: 595ms
50:	learn: 0.1265428	total: 121ms	remaining: 592ms
[LightGBM] [Info] Number of positive: 13699, number of negative: 4825
51:	learn: 0.1263684	total: 124ms	remaining: 590ms
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001248 seconds.
You 

,estimators,"[('xgb', ...), ('lgbm', ...), ...]"
,voting,'soft'
,weights,"[np.float64(0.9702549091619229), np.float64(0.9699849890551265), ...]"
,n_jobs,-1
,flatten_transform,True
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None


In [33]:
predictions = ensemble.predict(X_test)
reverse_mapping = {1: 'Extrovert', 0: 'Introvert'}
final_predictions_text = pd.Series(predictions).map(reverse_mapping)

submission_df = pd.DataFrame({'id': test_df['id'], 'Personality': final_predictions_text})
submission_df.to_csv('submission.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")
print("Submission file head:")
print(submission_df.head())


Submission file 'submission.csv' created successfully.
Submission file head:
      id Personality
0  18524   Extrovert
1  18525   Introvert
2  18526   Extrovert
3  18527   Extrovert
4  18528   Introvert


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [47]:
stacking = StackingClassifier(
    estimators=[
        ('xgb', best_xgb),
        ('lgbm', best_lgbm),
        ('cat', best_catboost)
    ],
    final_estimator=LogisticRegression(),
    cv=cv,
    n_jobs=-1,
    passthrough=False  # True = ajoute les features originales à l’input du méta-modèle
)

stacking.fit(X, y)
preds_stack = stacking.predict(X_test)

/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:44:25] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)
/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [19:44:25] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 13699, number of negative: 4825
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000740 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 18524, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739527 -> initscore=1.043512
[LightGBM] [Info] Start training from score 1.043512
0:	learn: 0.6129472	total: 59ms	remaining: 17.6s
1:	learn: 0.5456588	total: 61.4ms	remaining: 9.15s
2:	learn: 0.4866700	total: 64.2ms	remaining: 6.35s
3:	learn: 0.4376015	total: 66.5ms	remaining: 4.92s
4:	learn: 0.3952016	total: 69.3ms	remaining: 4.09s
5:	learn: 0.3603733	total: 72.1ms	remaining: 3.53s
6:	learn: 0.3295251	total: 74.2ms	remaining: 3.11s
7:	learn: 0.3023780	total: 76.7ms	remaining: 2.8s
8:	learn: 0.2792319	total: 79.3ms	remaining

[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:203: XGBoost is not compiled with CUDA support.
[19:44:27] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

[19:44:27] WARNING: 

[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10959, number of negative: 3860
[LightGBM] [Info] Number of positive: 10960, number of negative: 3860
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004318 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 74
[LightGBM] [Info] Number of data points in the train set: 14819, number of used features: 9
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004565 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.739524 -> initscore=1.04

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


183:	learn: 0.1161562	total: 2.65s	remaining: 1.67s
180:	learn: 0.1183599	total: 2.6s	remaining: 1.71s
178:	learn: 0.1172819	total: 2.44s	remaining: 1.65s
181:	learn: 0.1200762	total: 2.75s	remaining: 1.78s
184:	learn: 0.1161327	total: 2.65s	remaining: 1.65s
179:	learn: 0.1172426	total: 2.44s	remaining: 1.63s
181:	learn: 0.1183292	total: 2.61s	remaining: 1.69s
182:	learn: 0.1200380	total: 2.76s	remaining: 1.76s
185:	learn: 0.1161024	total: 2.66s	remaining: 1.63s
183:	learn: 0.1198948	total: 2.64s	remaining: 1.66s
182:	learn: 0.1182832	total: 2.62s	remaining: 1.68s
183:	learn: 0.1200040	total: 2.77s	remaining: 1.74s
180:	learn: 0.1172180	total: 2.46s	remaining: 1.62s
186:	learn: 0.1160752	total: 2.67s	remaining: 1.61s
184:	learn: 0.1198389	total: 2.65s	remaining: 1.65s
181:	learn: 0.1171859	total: 2.46s	remaining: 1.6s
183:	learn: 0.1182280	total: 2.63s	remaining: 1.66s
185:	learn: 0.1198097	total: 2.65s	remaining: 1.62s
184:	learn: 0.1199706	total: 2.77s	remaining: 1.73s
187:	learn: 0.

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


215:	learn: 0.1158692	total: 2.63s	remaining: 1.02s
221:	learn: 0.1169545	total: 2.8s	remaining: 985ms
219:	learn: 0.1187417	total: 2.95s	remaining: 1.07s
221:	learn: 0.1184372	total: 2.82s	remaining: 992ms
222:	learn: 0.1169233	total: 2.81s	remaining: 969ms
220:	learn: 0.1149854	total: 2.85s	remaining: 1.02s
216:	learn: 0.1158225	total: 2.64s	remaining: 1.01s
220:	learn: 0.1187081	total: 2.95s	remaining: 1.05s
222:	learn: 0.1184127	total: 2.83s	remaining: 977ms
221:	learn: 0.1149636	total: 2.85s	remaining: 1s
217:	learn: 0.1157927	total: 2.64s	remaining: 993ms
223:	learn: 0.1168845	total: 2.81s	remaining: 953ms
223:	learn: 0.1183749	total: 2.83s	remaining: 960ms
222:	learn: 0.1149276	total: 2.85s	remaining: 986ms
221:	learn: 0.1186453	total: 2.96s	remaining: 1.04s
218:	learn: 0.1157278	total: 2.64s	remaining: 978ms
224:	learn: 0.1168591	total: 2.81s	remaining: 938ms
224:	learn: 0.1183310	total: 2.83s	remaining: 945ms
223:	learn: 0.1149057	total: 2.86s	remaining: 970ms
222:	learn: 0.11

/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [48]:
preds_stack

array([1, 0, 1, ..., 1, 1, 0], shape=(6175,))

In [56]:
predictions = stacking.predict(X_test)
reverse_mapping = {1: 'Extrovert', 0: 'Introvert'}
final_predictions_text = pd.Series(predictions).map(reverse_mapping)

submission_df = pd.DataFrame({'id': test_df['id'], 'Personality': final_predictions_text})
submission_df.to_csv('submission_stacking.csv', index=False)

print("\nSubmission file 'submission.csv' created successfully.")
print("Submission file head:")
print(submission_df.head())


Submission file 'submission.csv' created successfully.
Submission file head:
      id Personality
0  18524   Extrovert
1  18525   Introvert
2  18526   Extrovert
3  18527   Extrovert
4  18528   Introvert


/opt/miniconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [44]:
preds_stack

,0,1,2,3,4,5,6,7,8,9,...,6165,6166,6167,6168,6169,6170,6171,6172,6173,6174
id,,,,,,,,,,,,,,,,,,,,,
18524,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
18525,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
18526,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
18527,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
18528,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24694,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
24695,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
24696,1,0,1,1,0,1,1,0,1,0,...,1,1,1,1,1,1,0,1,1,0
